In [1]:
import os
os.chdir("../../web_backend/")

In [2]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

from search import Search, GraphSearcher, TextEmbeddingSearcher, EmbeddingSearcher

In [3]:
from app import init_DMG, search_collection

In [5]:
def init_DMG():
    DMG_DIR = "./data/DMG"
    image_folder = DMG_DIR+"/images/"
    image_handler = ImageHandler("DMG", image_folder=image_folder, keep_prefix=False)

    time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump(DMG_DIR+"/dumps")
    print(time_stamp)

    dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp, language="nl")
    df = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path=DMG_DIR+"/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)

    kg_searcher = GraphSearcher(df)


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=None)
    concept_search = TextEmbeddingSearcher(sem_embs, name="concept-searcher")


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=32)
    sem_searcher = EmbeddingSearcher(sem_embs, name="semantic-searcher")
    
    viz_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/vitmae", loadXD=32)
    viz_searcher = EmbeddingSearcher(viz_embs, name="visual-searcher")

    s = Search([kg_searcher, sem_searcher, viz_searcher])
    return df, s, concept_search, sem_embs, sem_searcher

df, s, cs, sem_embs, sem_searcher = init_DMG()

2026-08-17


[GraphSearcher]: building graph...: 100%|████████████████████████████| 22337/22337 [00:02<00:00, 8394.02it/s]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
df[df.index.str.startswith("2012-0025")]

,object_URI,title,description,objectname_URI,objectname_label,subcollection_URI,subcollection_name,material_URI,material_label,part_label,...,thumb_width,thumb_height,filename,path,thumb_path,dominant_R,dominant_G,dominant_B,time,sort_rank
object_number,,,,,,,,,,,,,,,,,,,,,
2012-0025_00-35,https://data.designmuseumgent.be/v2/id/object/...,Servies met decor 'Etruria',"Deze groep serviesdelen, bestaande uit terrine...",https://data.designmuseumgent.be/v2/id/concept...,schaal&semi;sauskom&semi;terrine&semi;soepterr...,https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3521
2012-0025_01-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3522
2012-0025_02-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3523
2012-0025_03-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3524
2012-0025_04-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3525
2012-0025_05-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3526
2012-0025_06-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3527
2012-0025_07-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3528
2012-0025_08-35,https://data.designmuseumgent.be/v2/id/object/...,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,https://data.designmuseumgent.be/v2/id/concept...,bord (vaatwerk),https://data.designmuseumgent.be/v2/id/concept...,keramiek,https://data.designmuseumgent.be/v2/id/concept...,faïence fine,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1863/1914~,3529


---

In [11]:
duplicated = "2012-0025_06-35"

dup = """
Solifleur met passiebloem

Designer
    Ontwerper onbekend
Inventory number
    ID: 1987-1095
Description
    Glazen solifleur (vaas voor één bloem) met bolle buik en smalle hals met mondrand, vervaardigd in cameoglas. De basiskleur is opaak wit met een paarse vleug en het decor is in paarstinten uitgewerkt. De buik is gedecoreerd met een grote passiebloem, over de hals lopen ranken en knoppen. Het model van deze vaas wordt ook wel als 'banjo' aangeduid. De vaas bevat het signatuur 'Gallé' met Latijnse 'é' en dateert wellicht van na het overlijden van Emile Gallé.
    """


"""
Solifleur met passiebloem

Designer
    Ontwerper onbekend
Inventory number
    ID: 1987-1095
Description
    Glazen solifleur (vaas voor één bloem) met bolle buik en smalle hals met mondrand, vervaardigd in cameoglas. De basiskleur is opaak wit met een paarse vleug en het decor is in paarstinten uitgewerkt. De buik is gedecoreerd met een grote passiebloem, over de hals lopen ranken en knoppen. Het model van deze vaas wordt ook wel als 'banjo' aangeduid. De vaas bevat het signatuur 'Gallé' met Latijnse 'é' en dateert wellicht van na het overlijden van Emile Gallé.
"""

In [27]:
import requests as rq


def parse_id_list(id_list_str):
    try:
        return list(map(str.strip, id_list_str.split(",")))
    except ValueError:
        raise s


url = "http://127.0.0.1:8080/DMG_2026-08-17/search/order"
# url = "https://backend.dev.searcher.studio/DMG_2026-08-19/search/order"
params = {
    "model_ids": "graph-searcher,semantic-searcher,visual-searcher",
    "object_ids": "1999-0083_19-23,2012-0025_06-35,2009-0049_26-71",
    "limit": "20"
    
}

# url = "https://backend.dev.searcher.studio/DMG_2026-08-19/search/order?model_ids=graph-searcher,semantic-searcher,visual-searcher&object_ids=1999-0083_19-23,2012-0025_06-35,2009-0049_26-71&limit=20"

resp = rq.get(url, params=params)
resp.raise_for_status()

data = resp.json()

# https://backend.dev.searcher.studio/DMG_2026-08-19/search/order?model_ids=graph-searcher,semantic-searcher,visual-searcher&object_ids=


# https://backend.dev.searcher.studio/DMG_2026-08-19/search/order?
# model_ids=graph-searcher,semantic-searcher,visual-searcher&object_ids=1999-0083_19-23,2012-0025_06-35,2009-0049_26-71&limit=20


['1999-0083_19-23', '2012-0025_06-35', '2009-0049_26-71']

In [29]:
result_df = pd.DataFrame.from_records(data).set_index("inventory_number")

# result_df.loc[parse_id_list(params["object_ids"])]
result_df

,title,description,designer,producer,design_date,production_date,design_place,production_place,rights_attribution,image,order_index
inventory_number,,,,,,,,,,,
2009-0049_26-71,Bord (groot) van faïence,,,Faïenceries de Longwy,,na 1930,,Longwy,In Copyright,None,0
1999-0083_19-23,,,Wilhelm Wagenfeld,Jenaer Glaswerk Schott & Gen,1931 — 1931,na 1931,Jena,,In Copyright,None,1
2012-0025_06-35,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,Société Céramique,Société Céramique,1863 — ca. 1914,1863 — ca. 1914,Maastricht,Maastricht,In Copyright,None,2
2010-0028_62-72,Bord van porseleinen servies,Het servies werd uitgebracht onder licentie va...,onbekend,Porcelaine Lafarge & Cie,ca. 1970 — ca. 1970,,,Limoges,In Copyright,None,3
1998-0024_061-755,Bord van het servies 'Modus Vivendi' uit de in...,In 1997-1998 maakte Pieter Stockmans de instal...,Pieter Stockmans,Piet Stockmans; Indoor; Studio Pieter Stockmans,1992 — 1992,1997 — 1998,Genk,Genk; Amsterdam,In Copyright,None,4
1998-0024_050-755,Bord van het servies 'Modus Vivendi' uit de in...,In 1997-1998 maakte Pieter Stockmans de instal...,Pieter Stockmans,Piet Stockmans; Indoor; Studio Pieter Stockmans,1992 — 1992,1997 — 1998,Genk,Genk; Amsterdam,In Copyright,None,5
1990-0024_7-9,Servet bij het tafellaken 'Le bouclier noir',Deze servetten horen bij het tafellaken 'Le bo...,Emile-Jean Braun,Goeters - Ars et Labor; UCO,,1964 — 1964,Gent,Lokeren (Oost-Vlaanderen); Gent,In Copyright,None,6
1987-0985_16-89,Portoglas van het servies ‘H’ of ‘Hygpa’,Portoglas in helder glas als onderdeel van het...,Karel Petrus Cornelis de Bazel,Royal Leerdam Crystal,1918 — 1918,,Nederland,Leerdam,In Copyright,None,7
1997-0084_5-7,poly poly,,Rudi Respeel,Rudi Respeel,na 1997,na 1997,Sint-Lenaarts,,In Copyright,None,8


In [26]:
# def move_to_front(df, idx_labels):
#     rest = df.index.difference(idx_labels, sort=False)
#     return df.loc[list(idx_labels) + list(rest)]

def move_to_front_preserve_order(df, idx_labels):
    idx_labels = set(idx_labels)
    mask = df.index.isin(idx_labels)
    order = np.argsort(~mask, kind='stable')
    result = df.iloc[order]
    result.attrs = df.attrs.copy()  # explicit, version-independent
    return result

    
move_to_front_preserve_order(result_df, parse_id_list(params["object_ids"]))

,title,description,designer,producer,design_date,production_date,design_place,production_place,rights_attribution,image,order_index
inventory_number,,,,,,,,,,,
2009-0049_26-71,Bord (groot) van faïence,,,Faïenceries de Longwy,,na 1930,,Longwy,In Copyright,None,0
1999-0083_19-23,,,Wilhelm Wagenfeld,Jenaer Glaswerk Schott & Gen,1931 — 1931,na 1931,Jena,,In Copyright,None,1
2012-0025_06-35,Bord met decor 'Etruria',Dit bord is bedrukt met een decor van weelderi...,Société Céramique,Société Céramique,1863 — ca. 1914,1863 — ca. 1914,Maastricht,Maastricht,In Copyright,None,6
2010-0028_62-72,Bord van porseleinen servies,Het servies werd uitgebracht onder licentie va...,onbekend,Porcelaine Lafarge & Cie,ca. 1970 — ca. 1970,,,Limoges,In Copyright,None,2
1998-0024_061-755,Bord van het servies 'Modus Vivendi' uit de in...,In 1997-1998 maakte Pieter Stockmans de instal...,Pieter Stockmans,Piet Stockmans; Indoor; Studio Pieter Stockmans,1992 — 1992,1997 — 1998,Genk,Genk; Amsterdam,In Copyright,None,3
1998-0024_050-755,Bord van het servies 'Modus Vivendi' uit de in...,In 1997-1998 maakte Pieter Stockmans de instal...,Pieter Stockmans,Piet Stockmans; Indoor; Studio Pieter Stockmans,1992 — 1992,1997 — 1998,Genk,Genk; Amsterdam,In Copyright,None,4
1990-0024_7-9,Servet bij het tafellaken 'Le bouclier noir',Deze servetten horen bij het tafellaken 'Le bo...,Emile-Jean Braun,Goeters - Ars et Labor; UCO,,1964 — 1964,Gent,Lokeren (Oost-Vlaanderen); Gent,In Copyright,None,5
1987-0985_16-89,Portoglas van het servies ‘H’ of ‘Hygpa’,Portoglas in helder glas als onderdeel van het...,Karel Petrus Cornelis de Bazel,Royal Leerdam Crystal,1918 — 1918,,Nederland,Leerdam,In Copyright,None,7
1997-0084_5-7,poly poly,,Rudi Respeel,Rudi Respeel,na 1997,na 1997,Sint-Lenaarts,,In Copyright,None,8


---

In [71]:

result_df.inventory_number.value_counts()

inventory_number
1999-0083_19-23      4
2012-0025_06-35      1
2009-0049_26-71      1
1998-0024_061-755    1
1998-0024_050-755    1
1990-0024_7-9        1
1987-0985_16-89      1
1997-0084_5-7        1
3279                 1
2024-0063            1
2018-0367_0-5        1
1980-0289            1
3471                 1
1980-0117            1
1998-0024_159-755    1
1998-0024_110-755    1
2018-0002_1-7        1
4700_2-3             1
SCH-0453_0-4         1
Name: count, dtype: int64

In [63]:
pd.Series([d["inventory_number"] for d in data])

0       1999-0083_19-23
1       2012-0025_06-35
2       2012-0025_06-35
3       2009-0049_26-71
4       2012-0025_04-35
5         1987-1542_4-5
6       2012-0025_21-35
7       2012-0025_08-35
8       2012-0025_06-35
9       2012-0025_06-35
10      2012-0025_03-35
11      2012-0025_07-35
12      2012-0025_15-35
13      2012-0025_18-35
14      2012-0025_11-35
15      2012-0025_01-35
16      2012-0025_12-35
17      2012-0025_13-35
18    1998-0024_000-755
19      2012-0025_23-35
20        2014-0026_0-4
21      2009-0049_24-71
dtype: str

In [42]:
result_df.inventory_number.value_counts().sort_values()

inventory_number
1999-0083_19-23    1
2012-0025_06-35    1
2009-0049_26-71    1
5052               1
5055               1
5057_1-2           1
5057_2-2           1
5056               1
0977               1
0980               1
3988               1
0008               1
0832               1
0833               1
0834               1
0836               1
0843               1
0844               1
0847               1
0848               1
Name: count, dtype: int64

In [34]:
list(result_df.order_index.iloc[:300])

[7975,
 3526,
 7793,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 18

In [35]:
result_df[result_df.title == "Solifleur met passiebloem"]

,inventory_number,title,description,designer,producer,design_date,production_date,design_place,production_place,rights_attribution,image,order_index
5889,1987-1095,Solifleur met passiebloem,Glazen solifleur (vaas voor één bloem) met bol...,,Etablissements Emile Gallé,,ca. 1907 — ca. 1914,,Nancy,In Copyright,{'path': '/DMG/images/1987-1095/1987-1095$1.JP...,5889
